# Advanced Python 3.9 Problems with Complete Solutions

This notebook develops advanced, production-oriented exercises around four Python 3.9 additions:

1. Time-zone handling with `zoneinfo`
2. Multi-argument `math.gcd()` and the new `math.lcm()`
3. Dictionary union operators: `|` and `|=`
4. Exact prefix/suffix removal with `str.removeprefix()` and `str.removesuffix()`

Every problem includes a precise specification, a complete reference solution, executable checks, and design notes. The examples use deterministic data so results are reproducible.

## How to use this notebook

For each problem:

1. Read the requirements and edge cases.
2. Write your own solution in a new cell.
3. Run the supplied assertions.
4. Compare with the reference solution.
5. Add at least one test of your own.

**Runtime:** Python 3.9 or newer.

On Windows, `zoneinfo` may require:

```bash
python -m pip install tzdata
```

In [1]:
import math
import operator
import sys
from datetime import datetime, timedelta, timezone
from fractions import Fraction
from functools import reduce
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple
from zoneinfo import ZoneInfo, available_timezones

assert sys.version_info >= (3, 9), "This notebook requires Python 3.9+."
print("Python:", sys.version.split()[0])
print("Available IANA zones:", len(available_timezones()))

Python: 3.13.7
Available IANA zones: 598


## Best-practice checklist

- Store and compare instants in UTC.
- Convert to named IANA zones at display or business-rule boundaries.
- Do not assume that attaching `tzinfo` validates a local wall time.
- Use deterministic timestamps in tests.
- Prefer `math.gcd()` and `math.lcm()` to handwritten arithmetic.
- Treat dictionary union as a **shallow**, right-biased merge.
- Make precedence visible in code: `defaults | environment | command_line`.
- Use `removeprefix()` and `removesuffix()` for semantic substring removal.
- Do not use `strip()`, `lstrip()`, or `rstrip()` as substring removers.

# Part I — Advanced `zoneinfo` Problems

## Problem 1 — Convert one instant to many zones

Implement `project_instant(instant, zone_names)`.

Requirements:

- `instant` must be timezone-aware.
- Convert the same instant into every requested IANA zone.
- Preserve the input order of zone names.
- Reject a naive datetime with `ValueError`.
- Do not mutate the input.

In [2]:
def project_instant(
    instant: datetime,
    zone_names: Sequence[str],
) -> Dict[str, datetime]:
    """Project one aware instant into multiple named time zones."""
    if instant.tzinfo is None or instant.utcoffset() is None:
        raise ValueError("instant must be timezone-aware")

    return {
        zone_name: instant.astimezone(ZoneInfo(zone_name))
        for zone_name in zone_names
    }


reference_instant = datetime(2025, 3, 30, 0, 30, tzinfo=timezone.utc)

projected = project_instant(
    reference_instant,
    [
        "UTC",
        "Europe/Sofia",
        "Europe/London",
        "America/New_York",
        "Australia/Melbourne",
    ],
)

for zone_name, local_dt in projected.items():
    print(f"{zone_name:22} {local_dt.isoformat()}")

UTC                    2025-03-30T00:30:00+00:00
Europe/Sofia           2025-03-30T02:30:00+02:00
Europe/London          2025-03-30T00:30:00+00:00
America/New_York       2025-03-29T20:30:00-04:00
Australia/Melbourne    2025-03-30T11:30:00+11:00


In [3]:
assert list(projected) == [
    "UTC",
    "Europe/Sofia",
    "Europe/London",
    "America/New_York",
    "Australia/Melbourne",
]
assert all(
    value.astimezone(timezone.utc) == reference_instant
    for value in projected.values()
)

try:
    project_instant(datetime(2025, 1, 1, 12, 0), ["UTC"])
except ValueError as exc:
    assert "timezone-aware" in str(exc)
else:
    raise AssertionError("A naive datetime should be rejected.")

print("Problem 1 tests passed.")

Problem 1 tests passed.


### Why this works

All projected values represent the same instant. Their wall-clock readings differ, but each converts back to the original UTC timestamp.

## Problem 2 — Classify normal, ambiguous, and nonexistent local times

Implement `classify_local_time(local_naive, zone_name)` returning:

- `"normal"`
- `"ambiguous"`
- `"nonexistent"`

Constraints:

- The input datetime must be naive.
- Validate candidates by round-tripping through UTC.
- Use the PEP 495 `fold` flag.
- Do not hard-code offsets.

In [4]:
def classify_local_time(local_naive: datetime, zone_name: str) -> str:
    """Classify a naive wall time in an IANA time zone."""
    if local_naive.tzinfo is not None:
        raise ValueError("local_naive must not have tzinfo")

    zone = ZoneInfo(zone_name)
    valid_candidates: List[datetime] = []

    for fold in (0, 1):
        candidate = local_naive.replace(tzinfo=zone, fold=fold)
        round_trip = (
            candidate
            .astimezone(timezone.utc)
            .astimezone(zone)
            .replace(tzinfo=None)
        )
        if round_trip == local_naive:
            valid_candidates.append(candidate)

    if not valid_candidates:
        return "nonexistent"

    distinct_offsets = {
        candidate.utcoffset()
        for candidate in valid_candidates
    }
    return "ambiguous" if len(distinct_offsets) > 1 else "normal"


examples = {
    "normal": datetime(2024, 2, 1, 12, 0),
    "nonexistent": datetime(2024, 3, 10, 2, 30),
    "ambiguous": datetime(2024, 11, 3, 1, 30),
}

for expected, wall_time in examples.items():
    actual = classify_local_time(wall_time, "America/New_York")
    print(wall_time, "->", actual)
    assert actual == expected

2024-02-01 12:00:00 -> normal
2024-03-10 02:30:00 -> nonexistent
2024-11-03 01:30:00 -> ambiguous


In [5]:
assert classify_local_time(
    datetime(2024, 3, 31, 3, 30),
    "Europe/Sofia",
) == "nonexistent"

assert classify_local_time(
    datetime(2024, 10, 27, 3, 30),
    "Europe/Sofia",
) == "ambiguous"

assert classify_local_time(
    datetime(2024, 7, 15, 10, 0),
    "Europe/Sofia",
) == "normal"

print("Problem 2 tests passed.")

Problem 2 tests passed.


## Problem 3 — Resolve an ambiguous wall time explicitly

Implement `resolve_local_time(local_naive, zone_name, occurrence)`.

- `occurrence="earlier"` chooses the first occurrence.
- `occurrence="later"` chooses the second occurrence.
- Reject nonexistent local times.
- Normal local times have one real instant.
- Return a timezone-aware datetime.

In [6]:
def valid_local_candidates(
    local_naive: datetime,
    zone: ZoneInfo,
) -> List[datetime]:
    candidates: List[datetime] = []

    for fold in (0, 1):
        candidate = local_naive.replace(tzinfo=zone, fold=fold)
        round_trip = (
            candidate
            .astimezone(timezone.utc)
            .astimezone(zone)
            .replace(tzinfo=None)
        )
        if round_trip == local_naive:
            candidates.append(candidate)

    # Deduplicate equivalent normal-time candidates by UTC instant.
    unique_by_utc: Dict[datetime, datetime] = {}
    for candidate in candidates:
        unique_by_utc[candidate.astimezone(timezone.utc)] = candidate

    return [unique_by_utc[key] for key in sorted(unique_by_utc)]


def resolve_local_time(
    local_naive: datetime,
    zone_name: str,
    occurrence: str = "earlier",
) -> datetime:
    """Resolve a naive wall time to a real aware datetime."""
    if local_naive.tzinfo is not None:
        raise ValueError("local_naive must be naive")
    if occurrence not in {"earlier", "later"}:
        raise ValueError("occurrence must be 'earlier' or 'later'")

    candidates = valid_local_candidates(
        local_naive,
        ZoneInfo(zone_name),
    )

    if not candidates:
        raise ValueError(
            f"{local_naive!s} does not exist in {zone_name}"
        )

    if len(candidates) == 1:
        return candidates[0]

    return candidates[0] if occurrence == "earlier" else candidates[-1]


wall_time = datetime(2024, 11, 3, 1, 30)
first = resolve_local_time(wall_time, "America/New_York", "earlier")
second = resolve_local_time(wall_time, "America/New_York", "later")

print("Earlier:", first.isoformat(), "UTC:", first.astimezone(timezone.utc))
print("Later:  ", second.isoformat(), "UTC:", second.astimezone(timezone.utc))

Earlier: 2024-11-03T01:30:00-04:00 UTC: 2024-11-03 05:30:00+00:00
Later:   2024-11-03T01:30:00-05:00 UTC: 2024-11-03 06:30:00+00:00


In [7]:
assert first.fold == 0
assert second.fold == 1
assert (
    second.astimezone(timezone.utc)
    - first.astimezone(timezone.utc)
) == timedelta(hours=1)

try:
    resolve_local_time(
        datetime(2024, 3, 10, 2, 30),
        "America/New_York",
    )
except ValueError as exc:
    assert "does not exist" in str(exc)
else:
    raise AssertionError("A nonexistent local time must be rejected.")

print("Problem 3 tests passed.")

Problem 3 tests passed.


## Problem 4 — Build a recurring meeting that preserves wall-clock time

A Sofia team schedules a meeting every Monday at 09:30 local time. Generate `count` occurrences.

Return dictionaries containing:

- `local`
- `utc`
- `new_york`
- `melbourne`

The meeting must remain at 09:30 in Sofia even when the UTC offset changes. Do not generate the recurrence by repeatedly adding seven days to a UTC timestamp.

In [8]:
def weekly_wall_clock_schedule(
    first_local_naive: datetime,
    organizer_zone: str,
    count: int,
) -> List[Dict[str, datetime]]:
    if first_local_naive.tzinfo is not None:
        raise ValueError("first_local_naive must be naive")
    if count < 0:
        raise ValueError("count must be non-negative")

    rows: List[Dict[str, datetime]] = []

    for week in range(count):
        wall_time = first_local_naive + timedelta(weeks=week)
        local = resolve_local_time(wall_time, organizer_zone)
        utc_value = local.astimezone(timezone.utc)

        rows.append(
            {
                "local": local,
                "utc": utc_value,
                "new_york": utc_value.astimezone(
                    ZoneInfo("America/New_York")
                ),
                "melbourne": utc_value.astimezone(
                    ZoneInfo("Australia/Melbourne")
                ),
            }
        )

    return rows


meeting_rows = weekly_wall_clock_schedule(
    datetime(2024, 3, 18, 9, 30),
    "Europe/Sofia",
    4,
)

for row in meeting_rows:
    print(
        "Sofia:", row["local"].isoformat(),
        "| UTC:", row["utc"].isoformat(),
        "| New York:", row["new_york"].isoformat(),
        "| Melbourne:", row["melbourne"].isoformat(),
    )

Sofia: 2024-03-18T09:30:00+02:00 | UTC: 2024-03-18T07:30:00+00:00 | New York: 2024-03-18T03:30:00-04:00 | Melbourne: 2024-03-18T18:30:00+11:00
Sofia: 2024-03-25T09:30:00+02:00 | UTC: 2024-03-25T07:30:00+00:00 | New York: 2024-03-25T03:30:00-04:00 | Melbourne: 2024-03-25T18:30:00+11:00
Sofia: 2024-04-01T09:30:00+03:00 | UTC: 2024-04-01T06:30:00+00:00 | New York: 2024-04-01T02:30:00-04:00 | Melbourne: 2024-04-01T17:30:00+11:00
Sofia: 2024-04-08T09:30:00+03:00 | UTC: 2024-04-08T06:30:00+00:00 | New York: 2024-04-08T02:30:00-04:00 | Melbourne: 2024-04-08T16:30:00+10:00


In [9]:
assert len(meeting_rows) == 4
assert {
    (row["local"].hour, row["local"].minute)
    for row in meeting_rows
} == {(9, 30)}

assert meeting_rows[0]["local"].utcoffset() == timedelta(hours=2)
assert meeting_rows[-1]["local"].utcoffset() == timedelta(hours=3)
assert meeting_rows[0]["utc"].hour == 7
assert meeting_rows[-1]["utc"].hour == 6

print("Problem 4 tests passed.")

Problem 4 tests passed.


## Problem 5 — Normalize API timestamps to canonical UTC strings

Implement `canonical_utc(timestamp)`.

Requirements:

- Accept an aware datetime.
- Convert it to UTC.
- Return ISO 8601 text ending in `Z`.
- Include seconds.
- Preserve microseconds only when nonzero.
- Reject naive datetimes.

In [10]:
def canonical_utc(timestamp: datetime) -> str:
    if timestamp.tzinfo is None or timestamp.utcoffset() is None:
        raise ValueError("timestamp must be timezone-aware")

    utc_value = timestamp.astimezone(timezone.utc)
    timespec = "microseconds" if utc_value.microsecond else "seconds"
    return utc_value.isoformat(timespec=timespec).removesuffix("+00:00") + "Z"


assert canonical_utc(
    datetime(2025, 7, 1, 15, 45, tzinfo=ZoneInfo("Europe/Sofia"))
) == "2025-07-01T12:45:00Z"

assert canonical_utc(
    datetime(
        2025, 7, 1, 15, 45, 12, 125000,
        tzinfo=ZoneInfo("Europe/Sofia"),
    )
) == "2025-07-01T12:45:12.125000Z"

print("Problem 5 tests passed.")

Problem 5 tests passed.


# Part II — Advanced `math.gcd()` and `math.lcm()` Problems

## Problem 6 — Normalize an integer ratio of arbitrary length

Implement `normalize_ratio(values)`.

Examples:

- `(84, 126, 210)` becomes `(2, 3, 5)`.
- `(-12, 18, -30)` becomes `(-2, 3, -5)`.
- All-zero input remains all zero.

Use multi-argument `math.gcd()`, preserve signs, reject empty input, and reject booleans.

In [11]:
def normalize_ratio(values: Sequence[int]) -> Tuple[int, ...]:
    if not values:
        raise ValueError("values must not be empty")
    if any(
        isinstance(value, bool) or not isinstance(value, int)
        for value in values
    ):
        raise TypeError("all values must be plain integers")

    divisor = math.gcd(*values)
    if divisor == 0:
        return tuple(values)

    return tuple(value // divisor for value in values)


assert normalize_ratio((84, 126, 210)) == (2, 3, 5)
assert normalize_ratio((-12, 18, -30)) == (-2, 3, -5)
assert normalize_ratio((0, 0, 0)) == (0, 0, 0)
assert normalize_ratio((0, 15, 30)) == (0, 1, 2)

print("Problem 6 tests passed.")

Problem 6 tests passed.


## Problem 7 — Compute a synchronization period

Each integer is a recurring job period in seconds. Return the earliest positive time when all jobs align again.

Requirements:

- Positive integers only.
- Non-empty input.
- Use `math.lcm()`.
- Support an optional `maximum` guard that raises `OverflowError`.

In [12]:
def synchronization_period(
    periods: Sequence[int],
    maximum: Optional[int] = None,
) -> int:
    if not periods:
        raise ValueError("periods must not be empty")
    if any(
        isinstance(period, bool)
        or not isinstance(period, int)
        or period <= 0
        for period in periods
    ):
        raise ValueError("periods must contain positive integers")

    result = math.lcm(*periods)

    if maximum is not None and result > maximum:
        raise OverflowError(
            f"synchronization period {result} exceeds {maximum}"
        )

    return result


assert synchronization_period([6, 10, 15]) == 30
assert synchronization_period([8, 12, 18]) == 72
assert synchronization_period([1]) == 1

try:
    synchronization_period([97, 101, 103], maximum=100_000)
except OverflowError:
    pass
else:
    raise AssertionError("The maximum guard should have fired.")

print("Problem 7 tests passed.")

Problem 7 tests passed.


## Problem 8 — Find the largest equal shard size

A set of datasets must be divided into equally sized shards with no remainder.

Implement:

- `largest_common_shard_size(sizes)`
- `shard_counts(sizes)`

The first returns the greatest valid shard size. The second returns the number of shards for each dataset.

In [13]:
def largest_common_shard_size(sizes: Sequence[int]) -> int:
    if not sizes:
        raise ValueError("sizes must not be empty")
    if any(
        isinstance(size, bool)
        or not isinstance(size, int)
        or size <= 0
        for size in sizes
    ):
        raise ValueError("sizes must contain positive integers")

    return math.gcd(*sizes)


def shard_counts(sizes: Sequence[int]) -> Tuple[int, ...]:
    shard_size = largest_common_shard_size(sizes)
    return tuple(size // shard_size for size in sizes)


dataset_sizes = (1260, 1890, 3150, 4410)
common_size = largest_common_shard_size(dataset_sizes)
counts = shard_counts(dataset_sizes)

assert common_size == 630
assert counts == (2, 3, 5, 7)
assert all(
    size == common_size * count
    for size, count in zip(dataset_sizes, counts)
)

print("Common shard size:", common_size)
print("Shard counts:", counts)

Common shard size: 630
Shard counts: (2, 3, 5, 7)


## Problem 9 — Convert rational durations to exact integer ticks

Given `Fraction` durations, calculate the smallest shared denominator using `math.lcm()` and return:

```python
(ticks_per_unit, tick_counts)
```

For `[1/3, 1/4, 5/6]`, the result is `(12, (4, 3, 10))`.

In [14]:
def rational_ticks(
    durations: Sequence[Fraction],
) -> Tuple[int, Tuple[int, ...]]:
    if not durations:
        raise ValueError("durations must not be empty")
    if any(not isinstance(item, Fraction) for item in durations):
        raise TypeError("all durations must be Fraction instances")

    ticks_per_unit = math.lcm(
        *(item.denominator for item in durations)
    )
    tick_counts = tuple(
        item.numerator * (ticks_per_unit // item.denominator)
        for item in durations
    )
    return ticks_per_unit, tick_counts


durations = [
    Fraction(1, 3),
    Fraction(1, 4),
    Fraction(5, 6),
]
ticks_per_unit, tick_counts = rational_ticks(durations)

assert ticks_per_unit == 12
assert tick_counts == (4, 3, 10)
assert [
    Fraction(ticks, ticks_per_unit)
    for ticks in tick_counts
] == durations

print("Ticks per unit:", ticks_per_unit)
print("Tick counts:", tick_counts)

Ticks per unit: 12
Tick counts: (4, 3, 10)


## Problem 10 — Solve a repeating phase-alignment problem

Implement `first_phase_alignment(periods, phases)` to find the first non-negative integer `t` satisfying:

```text
t % period == phase % period
```

for every pair.

Search at most one complete joint cycle, `math.lcm(*periods)`. Return `None` when no solution exists.

In [15]:
def first_phase_alignment(
    periods: Sequence[int],
    phases: Sequence[int],
) -> Optional[int]:
    if not periods or len(periods) != len(phases):
        raise ValueError(
            "periods and phases must have equal non-zero lengths"
        )
    if any(
        isinstance(period, bool)
        or not isinstance(period, int)
        or period <= 0
        for period in periods
    ):
        raise ValueError("periods must be positive integers")
    if any(
        isinstance(phase, bool)
        or not isinstance(phase, int)
        for phase in phases
    ):
        raise ValueError("phases must be integers")

    cycle = math.lcm(*periods)
    normalized = [
        phase % period
        for period, phase in zip(periods, phases)
    ]

    for moment in range(cycle):
        if all(
            moment % period == phase
            for period, phase in zip(periods, normalized)
        ):
            return moment

    return None


assert first_phase_alignment([8, 12], [1, 5]) == 17
assert first_phase_alignment([4, 6], [1, 3]) == 9
assert first_phase_alignment([8, 12, 18], [1, 5, 11]) == 65

print("Problem 10 tests passed.")

Problem 10 tests passed.


# Part III — Advanced Dictionary Union Problems

## Problem 11 — Layer application configuration

Merge configuration layers with this precedence:

```text
defaults < environment < command line
```

Use dictionary unions. Inputs must remain unchanged, and the result must be a plain dictionary.

In [16]:
def build_config(
    defaults: Mapping[str, Any],
    environment: Mapping[str, Any],
    command_line: Mapping[str, Any],
) -> Dict[str, Any]:
    return dict(defaults) | dict(environment) | dict(command_line)


defaults = {
    "host": "127.0.0.1",
    "port": 8000,
    "debug": False,
    "workers": 2,
}
environment = {
    "port": 9000,
    "workers": 4,
    "region": "eu-central",
}
command_line = {
    "debug": True,
    "workers": 8,
}

config = build_config(defaults, environment, command_line)

assert config == {
    "host": "127.0.0.1",
    "port": 9000,
    "debug": True,
    "workers": 8,
    "region": "eu-central",
}
assert list(config) == [
    "host", "port", "debug", "workers", "region"
]
assert defaults["workers"] == 2
assert environment["workers"] == 4

print(config)
print("Key order:", list(config))

{'host': '127.0.0.1', 'port': 9000, 'debug': True, 'workers': 8, 'region': 'eu-central'}
Key order: ['host', 'port', 'debug', 'workers', 'region']


### Key-order rule

When the right dictionary overwrites an existing key, the value changes but the key keeps its earlier position. New keys are appended in right-side insertion order.

## Problem 12 — Explain where every winning value came from

Implement `merge_with_provenance(named_layers)`.

Return:

1. the merged dictionary,
2. a dictionary mapping each key to the layer that supplied its final value.

In [17]:
def merge_with_provenance(
    named_layers: Sequence[Tuple[str, Mapping[str, Any]]],
) -> Tuple[Dict[str, Any], Dict[str, str]]:
    merged: Dict[str, Any] = {}
    provenance: Dict[str, str] = {}

    for layer_name, layer in named_layers:
        current = dict(layer)
        merged |= current
        provenance |= {
            key: layer_name
            for key in current
        }

    return merged, provenance


merged_config, config_sources = merge_with_provenance(
    [
        ("defaults", defaults),
        ("environment", environment),
        ("cli", command_line),
    ]
)

assert merged_config == config
assert config_sources == {
    "host": "defaults",
    "port": "environment",
    "debug": "cli",
    "workers": "cli",
    "region": "environment",
}

print("Merged:", merged_config)
print("Sources:", config_sources)

Merged: {'host': '127.0.0.1', 'port': 9000, 'debug': True, 'workers': 8, 'region': 'eu-central'}
Sources: {'host': 'defaults', 'port': 'environment', 'debug': 'cli', 'workers': 'cli', 'region': 'environment'}


## Problem 13 — Strict merge that rejects conflicting values

Implement `strict_union(*mappings)`.

- Repeating a key with the same value is allowed.
- Repeating a key with a different value raises `ValueError`.
- Preserve first-seen key order.
- Return a new dictionary.

In [18]:
def strict_union(*mappings: Mapping[str, Any]) -> Dict[str, Any]:
    result: Dict[str, Any] = {}

    for mapping in mappings:
        for key, value in mapping.items():
            if key in result and result[key] != value:
                raise ValueError(
                    f"conflict for {key!r}: "
                    f"{result[key]!r} != {value!r}"
                )
            result |= {key: value}

    return result


assert strict_union(
    {"host": "db.internal", "port": 5432},
    {"port": 5432, "ssl": True},
) == {
    "host": "db.internal",
    "port": 5432,
    "ssl": True,
}

try:
    strict_union({"port": 5432}, {"port": 5433})
except ValueError as exc:
    assert "port" in str(exc)
else:
    raise AssertionError("A conflicting value must be rejected.")

print("Problem 13 tests passed.")

Problem 13 tests passed.


## Problem 14 — Demonstrate shallow merge, then implement deep merge

Dictionary union is shallow. Implement `deep_right_union(left, right)`:

- Recursively merge only when both values are mappings.
- Otherwise, the right value wins.
- Never mutate inputs.
- Return plain dictionaries.

In [19]:
def deep_right_union(
    left: Mapping[str, Any],
    right: Mapping[str, Any],
) -> Dict[str, Any]:
    result = dict(left)

    for key, right_value in right.items():
        left_value = result.get(key)

        if (
            isinstance(left_value, Mapping)
            and isinstance(right_value, Mapping)
        ):
            result[key] = deep_right_union(left_value, right_value)
        else:
            result[key] = right_value

    return result


left_nested = {
    "database": {
        "host": "db.internal",
        "port": 5432,
        "options": {
            "ssl": False,
            "timeout": 10,
        },
    },
    "debug": False,
}

right_nested = {
    "database": {
        "options": {
            "ssl": True,
        },
    },
}

shallow = left_nested | right_nested
deep = deep_right_union(left_nested, right_nested)

assert shallow["database"] == {"options": {"ssl": True}}
assert deep["database"] == {
    "host": "db.internal",
    "port": 5432,
    "options": {
        "ssl": True,
        "timeout": 10,
    },
}
assert left_nested["database"]["options"]["ssl"] is False

print("Shallow:", shallow)
print("Deep:", deep)

Shallow: {'database': {'options': {'ssl': True}}, 'debug': False}
Deep: {'database': {'host': 'db.internal', 'port': 5432, 'options': {'ssl': True, 'timeout': 10}}, 'debug': False}


## Problem 15 — Merge many dictionaries functionally

Implement `union_all_functional(mappings)` with `reduce()` and `operator.or_`.

Also implement an imperative `|=` version. Empty input must return `{}`.

In [20]:
def union_all_functional(
    mappings: Iterable[Mapping[str, Any]],
) -> Dict[str, Any]:
    dictionaries = [dict(mapping) for mapping in mappings]
    return reduce(operator.or_, dictionaries, {})


def union_all_imperative(
    mappings: Iterable[Mapping[str, Any]],
) -> Dict[str, Any]:
    result: Dict[str, Any] = {}
    for mapping in mappings:
        result |= dict(mapping)
    return result


layers = [
    {"a": 1, "shared": "first"},
    {"b": 2},
    {"shared": "last", "c": 3},
]

expected_union = {
    "a": 1,
    "shared": "last",
    "b": 2,
    "c": 3,
}

assert union_all_functional(layers) == expected_union
assert union_all_imperative(layers) == expected_union
assert union_all_functional([]) == {}

print("Problem 15 tests passed.")

Problem 15 tests passed.


# Part IV — Advanced String Prefix/Suffix Problems

## Problem 16 — Why `lstrip()` is not prefix removal

Predict the output before running the cell.

`lstrip(chars)` removes leading characters found in the supplied character set. It does not remove one exact substring.

In [21]:
sample = "(log) log: service started"

incorrect = sample.lstrip("(log) ")
correct = sample.removeprefix("(log) ")

print("Original:", sample)
print("lstrip:", incorrect)
print("removeprefix:", correct)

assert incorrect == ": service started"
assert correct == "log: service started"

Original: (log) log: service started
lstrip: : service started
removeprefix: log: service started


## Problem 17 — Normalize one optional prefix and suffix

Implement `normalize_export_name(name)`.

- Remove one leading `"export-"`.
- Remove one trailing `".csv"`.
- Do not remove repeated occurrences beyond one at each edge.
- Do not alter occurrences in the middle.
- The function should be idempotent for normal inputs.

In [22]:
def normalize_export_name(name: str) -> str:
    if not isinstance(name, str):
        raise TypeError("name must be a string")

    return name.removeprefix("export-").removesuffix(".csv")


examples = {
    "export-sales.csv": "sales",
    "sales.csv": "sales",
    "export-sales": "sales",
    "sales": "sales",
    "export-export-sales.csv.csv": "export-sales.csv",
    "my-export-sales.csv-backup": "my-export-sales.csv-backup",
}

for raw, expected in examples.items():
    actual = normalize_export_name(raw)
    print(f"{raw!r:34} -> {actual!r}")
    assert actual == expected

assert normalize_export_name(
    normalize_export_name("export-sales.csv")
) == "sales"

print("Problem 17 tests passed.")

'export-sales.csv'                 -> 'sales'
'sales.csv'                        -> 'sales'
'export-sales'                     -> 'sales'
'sales'                            -> 'sales'
'export-export-sales.csv.csv'      -> 'export-sales.csv'
'my-export-sales.csv-backup'       -> 'my-export-sales.csv-backup'
Problem 17 tests passed.


## Problem 18 — Parse tagged log lines safely

Valid exact prefixes are:

- `"(debug) "`
- `"(info) "`
- `"(warning) "`
- `"(error) "`

Implement `parse_log_line(line)` returning `(level, message)`. Preserve the remainder unchanged and reject unknown prefixes.

In [23]:
LOG_PREFIXES = {
    "(debug) ": "DEBUG",
    "(info) ": "INFO",
    "(warning) ": "WARNING",
    "(error) ": "ERROR",
}


def parse_log_line(line: str) -> Tuple[str, str]:
    if not isinstance(line, str):
        raise TypeError("line must be a string")

    for prefix, level in LOG_PREFIXES.items():
        if line.startswith(prefix):
            return level, line.removeprefix(prefix)

    raise ValueError("unknown log prefix")


assert parse_log_line(
    "(info) service started"
) == ("INFO", "service started")

assert parse_log_line(
    "(error) (error) nested text"
) == ("ERROR", "(error) nested text")

try:
    parse_log_line("info: service started")
except ValueError:
    pass
else:
    raise AssertionError("Unknown syntax should be rejected.")

print("Problem 18 tests passed.")

Problem 18 tests passed.


## Problem 19 — Remove repeated wrappers deliberately

Implement `remove_all_prefixes(text, prefix, maximum=None)`.

- Remove repeated exact prefixes.
- Reject an empty prefix.
- `maximum=None` means unlimited removals.
- A non-negative integer limits removals.
- Return `(cleaned_text, removal_count)`.

In [24]:
def remove_all_prefixes(
    text: str,
    prefix: str,
    maximum: Optional[int] = None,
) -> Tuple[str, int]:
    if prefix == "":
        raise ValueError("prefix must not be empty")
    if maximum is not None:
        if (
            isinstance(maximum, bool)
            or not isinstance(maximum, int)
            or maximum < 0
        ):
            raise ValueError(
                "maximum must be a non-negative integer or None"
            )

    cleaned = text
    count = 0

    while cleaned.startswith(prefix):
        if maximum is not None and count >= maximum:
            break
        cleaned = cleaned.removeprefix(prefix)
        count += 1

    return cleaned, count


assert remove_all_prefixes(
    "tmp-tmp-tmp-report",
    "tmp-",
) == ("report", 3)

assert remove_all_prefixes(
    "tmp-tmp-tmp-report",
    "tmp-",
    maximum=2,
) == ("tmp-report", 2)

assert remove_all_prefixes(
    "report",
    "tmp-",
) == ("report", 0)

print("Problem 19 tests passed.")

Problem 19 tests passed.


## Problem 20 — Canonicalize resource identifiers

A resource identifier may look like:

```text
urn:job:<name>:v1
```

Implement `canonical_job_name(raw)`.

- Remove one optional `"urn:job:"` prefix.
- Remove one optional `":v1"` suffix.
- Trim surrounding whitespace.
- Lowercase the result.
- Replace internal spaces with hyphens.
- Reject an empty final name.

In [25]:
def canonical_job_name(raw: str) -> str:
    if not isinstance(raw, str):
        raise TypeError("raw must be a string")

    cleaned = (
        raw
        .strip()
        .removeprefix("urn:job:")
        .removesuffix(":v1")
        .strip()
        .lower()
        .replace(" ", "-")
    )

    if not cleaned:
        raise ValueError("job name is empty")

    return cleaned


assert canonical_job_name(
    " urn:job:Daily Revenue:v1 "
) == "daily-revenue"

assert canonical_job_name(
    "Daily Revenue"
) == "daily-revenue"

assert canonical_job_name(
    "urn:job:urn:job:test:v1:v1"
) == "urn:job:test:v1"

print("Problem 20 tests passed.")

Problem 20 tests passed.


# Part V — Integrated Capstone

## Problem 21 — Multi-region deployment scheduler

Build a deployment plan using all four feature areas.

The function must:

1. Merge configuration layers with `|`.
2. Normalize a job identifier with `removeprefix()` and `removesuffix()`.
3. Compute a full repeating cycle with `math.lcm()`.
4. Compute a scheduling quantum with `math.gcd()`.
5. Resolve a local Sofia start time with `zoneinfo`.
6. Return UTC runs and regional projections.
7. Keep every generated instant timezone-aware.

In [26]:
def build_deployment_plan(
    default_config: Mapping[str, Any],
    environment_config: Mapping[str, Any],
    override_config: Mapping[str, Any],
    raw_job_name: str,
    first_local_naive: datetime,
    organizer_zone: str,
    display_zones: Sequence[str],
    run_count: int = 6,
) -> Dict[str, Any]:
    config = (
        dict(default_config)
        | dict(environment_config)
        | dict(override_config)
    )

    periods = tuple(config["periods_minutes"])
    if not periods or any(
        isinstance(period, bool)
        or not isinstance(period, int)
        or period <= 0
        for period in periods
    ):
        raise ValueError(
            "periods_minutes must contain positive integers"
        )

    if run_count < 0:
        raise ValueError("run_count must be non-negative")

    job_name = (
        raw_job_name
        .strip()
        .removeprefix("job:")
        .removesuffix(".task")
        .strip()
    )
    if not job_name:
        raise ValueError("job name is empty")

    quantum_minutes = math.gcd(*periods)
    cycle_minutes = math.lcm(*periods)

    first_local = resolve_local_time(
        first_local_naive,
        organizer_zone,
        occurrence=config.get("ambiguous_occurrence", "earlier"),
    )
    first_utc = first_local.astimezone(timezone.utc)

    runs: List[Dict[str, Any]] = []

    for index in range(run_count):
        utc_run = first_utc + timedelta(
            minutes=index * quantum_minutes
        )

        projections = {
            zone_name: utc_run.astimezone(ZoneInfo(zone_name))
            for zone_name in display_zones
        }

        runs.append(
            {
                "index": index,
                "utc": utc_run,
                "canonical_utc": canonical_utc(utc_run),
                "regions": projections,
            }
        )

    return {
        "job": job_name,
        "config": config,
        "quantum_minutes": quantum_minutes,
        "cycle_minutes": cycle_minutes,
        "runs": runs,
    }


default_deployment = {
    "periods_minutes": [12, 18, 30],
    "ambiguous_occurrence": "earlier",
    "dry_run": True,
}
production_deployment = {
    "dry_run": False,
    "region": "eu",
}
manual_override = {
    "periods_minutes": [8, 12, 18],
}

deployment_plan = build_deployment_plan(
    default_deployment,
    production_deployment,
    manual_override,
    raw_job_name=" job:Revenue Snapshot.task ",
    first_local_naive=datetime(2024, 10, 27, 3, 30),
    organizer_zone="Europe/Sofia",
    display_zones=[
        "Europe/Sofia",
        "America/New_York",
        "Australia/Melbourne",
    ],
    run_count=8,
)

print("Job:", deployment_plan["job"])
print("Quantum:", deployment_plan["quantum_minutes"], "minutes")
print("Cycle:", deployment_plan["cycle_minutes"], "minutes")
print("Config:", deployment_plan["config"])
print()

for run in deployment_plan["runs"]:
    region_text = " | ".join(
        f"{name}={value.isoformat()}"
        for name, value in run["regions"].items()
    )
    print(
        f"{run['index']:2d} "
        f"{run['canonical_utc']} | "
        f"{region_text}"
    )

Job: Revenue Snapshot
Quantum: 2 minutes
Cycle: 72 minutes
Config: {'periods_minutes': [8, 12, 18], 'ambiguous_occurrence': 'earlier', 'dry_run': False, 'region': 'eu'}

 0 2024-10-27T00:30:00Z | Europe/Sofia=2024-10-27T03:30:00+03:00 | America/New_York=2024-10-26T20:30:00-04:00 | Australia/Melbourne=2024-10-27T11:30:00+11:00
 1 2024-10-27T00:32:00Z | Europe/Sofia=2024-10-27T03:32:00+03:00 | America/New_York=2024-10-26T20:32:00-04:00 | Australia/Melbourne=2024-10-27T11:32:00+11:00
 2 2024-10-27T00:34:00Z | Europe/Sofia=2024-10-27T03:34:00+03:00 | America/New_York=2024-10-26T20:34:00-04:00 | Australia/Melbourne=2024-10-27T11:34:00+11:00
 3 2024-10-27T00:36:00Z | Europe/Sofia=2024-10-27T03:36:00+03:00 | America/New_York=2024-10-26T20:36:00-04:00 | Australia/Melbourne=2024-10-27T11:36:00+11:00
 4 2024-10-27T00:38:00Z | Europe/Sofia=2024-10-27T03:38:00+03:00 | America/New_York=2024-10-26T20:38:00-04:00 | Australia/Melbourne=2024-10-27T11:38:00+11:00
 5 2024-10-27T00:40:00Z | Europe/Sofia=2

In [27]:
assert deployment_plan["job"] == "Revenue Snapshot"
assert deployment_plan["quantum_minutes"] == 2
assert deployment_plan["cycle_minutes"] == 72

assert deployment_plan["config"] == {
    "periods_minutes": [8, 12, 18],
    "ambiguous_occurrence": "earlier",
    "dry_run": False,
    "region": "eu",
}

assert len(deployment_plan["runs"]) == 8
assert all(
    run["utc"].tzinfo is not None
    for run in deployment_plan["runs"]
)
assert all(
    run["canonical_utc"].endswith("Z")
    for run in deployment_plan["runs"]
)

assert default_deployment["dry_run"] is True
assert production_deployment["dry_run"] is False

print("Capstone tests passed.")

Capstone tests passed.


# Part VI — Additional Challenge Problems

These extra problems intentionally omit complete solutions so you can continue practicing after studying the references above.

## Challenge A — Time-zone transition report

Given a zone and a UTC date range, identify every point where the UTC offset changes.

Suggested approach:

- iterate in coarse UTC steps,
- detect a changed offset after conversion,
- use binary search to narrow the transition,
- return the old offset, new offset, UTC transition, and local readings.

## Challenge B — Bounded least common multiple

Implement `bounded_lcm(values, limit)` without constructing a needlessly large intermediate result.

Use:

```python
next_lcm = current // math.gcd(current, value) * value
```

Check the limit before accepting each new value.

## Challenge C — Merge policy matrix

Design per-key merge policies:

- `"right"`: right wins,
- `"left"`: left wins,
- `"error"`: conflict raises,
- `"append"`: lists concatenate,
- `"deep"`: nested mappings merge recursively.

## Challenge D — Prefix/suffix transformation pipeline

Support rules that:

- remove one prefix,
- remove all repeated prefixes,
- remove one suffix,
- remove all repeated suffixes,
- add a missing prefix,
- add a missing suffix.

Produce an audit trail showing the text after every transformation.

# Final Review

You should now be able to:

- convert and validate timezone-aware datetimes,
- handle DST gaps and folds,
- preserve local wall-clock recurrence rules,
- compute multi-value GCDs and LCMs,
- build exact rational schedules,
- reason about dictionary-union precedence and insertion order,
- distinguish shallow and deep merging,
- remove semantic prefixes and suffixes safely,
- and combine all four feature families in production-style code.